# Count Graph
- Reload optimal composite for each batch / balance (6 combinations in total)
- Generate combined ranking for the subnetworks based on the specifications of the chosen composite (chosen centralities for high and low + applied weight) (both weighted and with threshold)
- Calculate correlation (2 per network, weighted and threshold) against each ground truth (4 corr. Scores)
- For each group of networks (unbalanced+50 cutoff, unbalanced+100 cutoff…etc…), generate graph counting the number of overall best-correlation of individual centralities to gt, including the count of the times that the composite metric outperforms the individual centralities in that network

In [21]:
import pandas as pd
import os
import json
from collections import Counter


In [22]:
def analyze_best_centralities(base_path: str = 'results/merged-subarticles-edges/importance-merged'):
    """
    Analyze best centralities and composite rankings across all analysis_results.json files.
    Tracks best weight and threshold combinations for each network.
    """    
    
    # Initialize counters for basic centralities
    best_high_counts = Counter()
    best_low_counts = Counter()
    best_overall_counts = Counter()
    
    # Track best composite configurations per network
    class CompositeResult:
        def __init__(self, optimized_for, weight_or_threshold, corr_importance, corr_doctypebranch, high_centrality, low_centrality):
            self.optimized_for = optimized_for
            self.value = weight_or_threshold
            self.corr_importance = corr_importance
            self.corr_doctypebranch = corr_doctypebranch
            self.avg_corr = (corr_importance + corr_doctypebranch) / 2
            self.high_centrality = high_centrality
            self.low_centrality = low_centrality
    
    # Dictionary to store best results for each network
    network_results = {}
    
    for root, dirs, files in os.walk(base_path):
        if 'comparisons' in root:
            continue
            
        if 'analysis_results.json' in files:
            try:
                network_name = os.path.basename(root)
                with open(os.path.join(root, 'analysis_results.json'), 'r') as f:
                    results = json.load(f)
                
                # Extract best basic centralities
                for ground_truth, analysis in results.get('ground_truth_analysis', {}).items():
                    best_centralities = analysis.get('best_centralities', {})
                    if 'high' in best_centralities:
                        best_high_counts[best_centralities['high']] += 1
                    if 'low' in best_centralities:
                        best_low_counts[best_centralities['low']] += 1
                    if 'overall' in best_centralities:
                        best_overall_counts[best_centralities['overall']] += 1
                
                # Track weight and threshold results for this network
                weight_results = []
                threshold_results = []
                
                ground_truth_analysis = results.get('ground_truth_analysis', {})
                
                # For each ground truth that was used for optimization
                for optimized_for, analysis in ground_truth_analysis.items():
                    weight = analysis.get('weight')
                    threshold = analysis.get('threshold')
                    correlations = analysis.get('correlations', {})
                    best_centralities = analysis.get('best_centralities', {})
                    
                    # Get the centralities used for this ground truth
                    high_centrality = best_centralities.get('high')
                    low_centrality = best_centralities.get('low')
                    
                    # Find correlations for both methods
                    weight_importance_corr = None
                    weight_doctypebranch_corr = None
                    threshold_importance_corr = None
                    threshold_doctypebranch_corr = None
                    
                    for pair, corr in correlations.items():
                        pair_str = str(pair)
                        if f"composite_ranking_{optimized_for}_weight_composite_ranking" in pair_str:
                            if "'importance'" in pair_str:
                                weight_importance_corr = corr
                            elif "'doctypebranch'" in pair_str:
                                weight_doctypebranch_corr = corr
                        elif f"composite_ranking_{optimized_for}_threshold_composite_ranking" in pair_str:
                            if "'importance'" in pair_str:
                                threshold_importance_corr = corr
                            elif "'doctypebranch'" in pair_str:
                                threshold_doctypebranch_corr = corr
                    
                    # Add results if we found both correlations
                    if weight_importance_corr is not None and weight_doctypebranch_corr is not None:
                        weight_results.append(
                            CompositeResult(
                                optimized_for,
                                weight,
                                weight_importance_corr,
                                weight_doctypebranch_corr,
                                high_centrality,
                                low_centrality
                            )
                        )
                    
                    if threshold_importance_corr is not None and threshold_doctypebranch_corr is not None:
                        threshold_results.append(
                            CompositeResult(
                                optimized_for,
                                threshold,
                                threshold_importance_corr,
                                threshold_doctypebranch_corr,
                                high_centrality,
                                low_centrality
                            )
                        )
                
                # Find best weight and threshold configurations for this network
                if weight_results:
                    best_weight = max(weight_results, key=lambda x: x.avg_corr)
                else:
                    best_weight = None
                    
                if threshold_results:
                    best_threshold = max(threshold_results, key=lambda x: x.avg_corr)
                else:
                    best_threshold = None
                
                # Store results for this network
                network_results[network_name] = {
                    'weight': best_weight,
                    'threshold': best_threshold
                }
                
                # Print results for this network
                print(f"\nNetwork: {network_name}")
                if best_weight:
                    print(f"Best Weight Configuration:")
                    print(f"  Optimized for: {best_weight.optimized_for}")
                    print(f"  Weight: {best_weight.value}")
                    print(f"  High centrality: {best_weight.high_centrality}")
                    print(f"  Low centrality: {best_weight.low_centrality}")
                    print(f"  Correlation with importance: {best_weight.corr_importance:.4f}")
                    print(f"  Correlation with doctypebranch: {best_weight.corr_doctypebranch:.4f}")
                    print(f"  Average correlation: {best_weight.avg_corr:.4f}")
                
                if best_threshold:
                    print(f"Best Threshold Configuration:")
                    print(f"  Optimized for: {best_threshold.optimized_for}")
                    print(f"  Threshold: {best_threshold.value}")
                    print(f"  High centrality: {best_threshold.high_centrality}")
                    print(f"  Low centrality: {best_threshold.low_centrality}")
                    print(f"  Correlation with importance: {best_threshold.corr_importance:.4f}")
                    print(f"  Correlation with doctypebranch: {best_threshold.corr_doctypebranch:.4f}")
                    print(f"  Average correlation: {best_threshold.avg_corr:.4f}")
                
            except Exception as e:
                print(f"Error processing {root}: {str(e)}")
                continue
    
    # Create summary DataFrame    
    summary = pd.DataFrame({
        'High Centrality': pd.Series(best_high_counts),
        'Low Centrality': pd.Series(best_low_counts),
        'Overall Best': pd.Series(best_overall_counts)
    })
    
    # Save summary to CSV
    output_path = os.path.join(base_path, 'centrality_summary.csv')
    summary.to_csv(output_path)
    
    # Print final summary
    print("\n=== Final Summary ===")
    print("\nBest High Centrality Counts:")
    for centrality, count in best_high_counts.most_common():
        print(f"{centrality}: {count}")
        
    print("\nBest Low Centrality Counts:")
    for centrality, count in best_low_counts.most_common():
        print(f"{centrality}: {count}")
        
    print("\nBest Overall Centrality Counts:")
    for centrality, count in best_overall_counts.most_common():
        print(f"{centrality}: {count}")
    
    return {
        'best_high_counts': best_high_counts,
        'best_low_counts': best_low_counts,
        'best_overall_counts': best_overall_counts,
        'network_results': network_results,
        'summary_df': summary
    }

In [11]:
results = analyze_best_centralities('results/fixed-merged-subarticles-edges/importance-merged')
print(results)


Network: balanced-doctypebranch
Best Weight Configuration:
  Optimized for: doctypebranch
  Weight: 95.0
  High centrality: in_degree_centrality
  Low centrality: relative_in_degree_centrality
  Correlation with importance: 0.7007
  Correlation with doctypebranch: 0.6263
  Average correlation: 0.6635
Best Threshold Configuration:
  Optimized for: importance
  Threshold: 82
  High centrality: degree_centrality
  Low centrality: core_number
  Correlation with importance: 0.7045
  Correlation with doctypebranch: 0.6278
  Average correlation: 0.6661

Network: balanced-importance
Best Weight Configuration:
  Optimized for: importance
  Weight: 50.0
  High centrality: pagerank
  Low centrality: in_degree_centrality
  Correlation with importance: 0.5240
  Correlation with doctypebranch: 0.4531
  Average correlation: 0.4885
Best Threshold Configuration:
  Optimized for: importance
  Threshold: 99
  High centrality: pagerank
  Low centrality: in_degree_centrality
  Correlation with importance:

In [23]:
from scipy import stats
import numpy as np
import pandas as pd

# Define the columns we care about
CENTRALITY_MEASURES = [
    'betweenness_centrality',
    'closeness_centrality', 
    'core_number',
    'degree_centrality',
    'disruption',
    'eigenvector_centrality',
    'harmonic_centrality',
    'hits_authority',
    'hits_hub',
    'in_degree_centrality',
    'out_degree_centrality',
    'pagerank',
    'relative_in_degree_centrality'
]

GROUND_TRUTHS = ['importance', 'doctypebranch']

def check_significance(df, columns_to_check, confidence=0.99):
    """
    Run Kendall's tau test on each centrality measure
    
    Args:
        df: DataFrame with centrality scores
        confidence: Confidence level (default 0.99 for 99%)
    """
    results = {}
    alpha = 1 - confidence
    
    for column in columns_to_check:
        if column not in ['ecli']:  # Skip non-centrality columns
            # Remove any NaN values
            data = df[column].dropna()
            
            # Perform Kendall's tau test
            tau, p_value = stats.kendalltau(np.arange(len(data)), data)
            
            # Check if significant at given confidence level
            is_significant = p_value < alpha
            
            results[column] = {
                'trend': tau,  # positive tau means increasing trend
                'p_value': p_value,
                'significant': is_significant
            }
    
    return results

# Load total_df from unbalanced results
import pandas as pd

total_df = pd.read_csv('results/merged-subarticles-edges/importance-merged/unbalanced/total_df.csv')

# Run test on your data
print("CENTRALITY MEASURES:")
significance_results = check_significance(total_df, CENTRALITY_MEASURES)

for metric, result in significance_results.items():
    print(f"\n{metric}:")
    print(f"Trend: {'Increasing' if result['trend'] > 0 else 'Decreasing'} (tau = {result['trend']:.4f})")
    print(f"P-value: {result['p_value']:.4f}")
    print(f"Significant at 99% confidence: {'Yes' if result['significant'] else 'No'}")

# Check value ranges only for centrality measures
print("\nVALUE RANGES FOR CENTRALITY MEASURES:")
for column in CENTRALITY_MEASURES:
    print(f"\n{column}:")
    print(f"Min: {total_df[column].min():.4f}")
    print(f"Max: {total_df[column].max():.4f}")

CENTRALITY MEASURES:

betweenness_centrality:
Trend: Decreasing (tau = -0.0224)
P-value: 0.0000
Significant at 99% confidence: Yes

closeness_centrality:
Trend: Decreasing (tau = -0.2001)
P-value: 0.0000
Significant at 99% confidence: Yes

core_number:
Trend: Increasing (tau = 0.1634)
P-value: 0.0000
Significant at 99% confidence: Yes

degree_centrality:
Trend: Increasing (tau = 0.1396)
P-value: 0.0000
Significant at 99% confidence: Yes

disruption:
Trend: Decreasing (tau = -0.0415)
P-value: 0.0000
Significant at 99% confidence: Yes

eigenvector_centrality:
Trend: Decreasing (tau = -0.1967)
P-value: 0.0000
Significant at 99% confidence: Yes

harmonic_centrality:
Trend: Decreasing (tau = -0.1965)
P-value: 0.0000
Significant at 99% confidence: Yes

hits_authority:
Trend: Decreasing (tau = -0.1012)
P-value: 0.0000
Significant at 99% confidence: Yes

hits_hub:
Trend: Increasing (tau = 0.1299)
P-value: 0.0000
Significant at 99% confidence: Yes

in_degree_centrality:
Trend: Decreasing (tau =

/var/folders/p6/pvd52f996qs30t8lkkf2qh0r0000gn/T/ipykernel_19328/2501104286.py:57: DtypeWarning: Columns (11,18) have mixed types. Specify dtype option on import or set low_memory=False.
  total_df = pd.read_csv('results/merged-subarticles-edges/importance-merged/unbalanced/total_df.csv')


In [24]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
total_df = pd.read_csv('results/fixed-merged-subarticles-edges/importance-merged/unbalanced/total_df.csv', low_memory=False)

# Calculate correlation and p-value between ground truths
correlation, p_value = stats.spearmanr(total_df['importance'], total_df['doctypebranch'])

# Create a small correlation matrix with p-values
ground_truths = ['importance', 'doctypebranch']
corr_matrix = pd.DataFrame(np.zeros((2, 2)), columns=ground_truths, index=ground_truths)
p_value_matrix = pd.DataFrame(np.zeros((2, 2)), columns=ground_truths, index=ground_truths)

print(f"Spearman correlation: {correlation:.6f}")
print(f"P-value: {p_value:.6f}")

for i in ground_truths:
    for j in ground_truths:
        corr, p = stats.spearmanr(total_df[i], total_df[j])
        corr_matrix.loc[i, j] = corr
        p_value_matrix.loc[i, j] = p

# Create markdown table
print("### Ground Truth Correlation Analysis\n")
print("#### Correlation Matrix:\n")
print("| Ground Truth | Importance | Doctypebranch |")
print("|-------------|------------|---------------|")
for idx in corr_matrix.index:
    row = [idx]
    for col in corr_matrix.columns:
        corr = corr_matrix.loc[idx, col]
        p = p_value_matrix.loc[idx, col]
        # Add significance stars
        stars = ''
        if p <= 0.001:
            stars = '***'
        elif p <= 0.01:
            stars = '**'
        elif p <= 0.05:
            stars = '*'
        row.append(f"{corr:.4f}{stars}")
    print(f"| {' | '.join(row)} |")

print("\n*p < 0.05, **p < 0.01, ***p < 0.001")

Spearman correlation: 0.393867
P-value: 0.000000
### Ground Truth Correlation Analysis

#### Correlation Matrix:

| Ground Truth | Importance | Doctypebranch |
|-------------|------------|---------------|
| importance | 1.0000*** | 0.3939*** |
| doctypebranch | 0.3939*** | 1.0000*** |

*p < 0.05, **p < 0.01, ***p < 0.001


In [9]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# Load the data
total_df = pd.read_csv('results/fixed-merged-subarticles-edges/importance-merged/unbalanced/total_df.csv', low_memory=False)

# Calculate correlation and p-value between ground truths
correlation, p_value = stats.spearmanr(total_df['importance'], total_df['doctypebranch'])

print(f"Correlation between ground truths:")
print(f"Spearman correlation: {correlation:.4f}")
print(f"P-value: {p_value:.4f}")

# Create correlation matrix for all measures
measures = ['importance', 'doctypebranch'] + CENTRALITY_MEASURES
corr_matrix = total_df[measures].corr(method='spearman')

# Calculate p-values matrix
def calculate_pvalue_matrix(df, measures):
    p_matrix = np.zeros((len(measures), len(measures)))
    for i, measure1 in enumerate(measures):
        for j, measure2 in enumerate(measures):
            _, p_value = stats.spearmanr(df[measure1], df[measure2])
            p_matrix[i, j] = p_value
    return pd.DataFrame(p_matrix, index=measures, columns=measures)

p_matrix = calculate_pvalue_matrix(total_df, measures)

# Create significance level indicators
def get_significance_markers(p_matrix):
    markers = p_matrix.copy()
    markers[p_matrix > 0.05] = ''
    markers[(p_matrix <= 0.05) & (p_matrix > 0.01)] = '*'    # p < 0.05
    markers[(p_matrix <= 0.01) & (p_matrix > 0.001)] = '**'  # p < 0.01
    markers[p_matrix <= 0.001] = '***'                       # p < 0.001
    return markers

significance_markers = get_significance_markers(p_matrix)

# Manually reorder measures to create smoother visual gradient
# Group similar centralities together based on their typical correlation patterns
ordered_measures = [
    'importance',
    'doctypebranch',
    'disruption',
    'hits_hub',
    'out_degree_centrality',
    'betweenness_centrality',
    'core_number',
    'degree_centrality',
    'hits_authority',
    'in_degree_centrality',
    'relative_in_degree_centrality',
    'pagerank',
    'eigenvector_centrality',
    'harmonic_centrality',
    'closeness_centrality'
]

# Reorder the correlation matrix
corr_matrix_ordered = corr_matrix.loc[ordered_measures, ordered_measures]

# Plot correlation matrix with manual ordering
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix_ordered, annot=True, cmap='RdBu', vmin=-1, vmax=1, center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

Correlation between ground truths:
Spearman correlation: 0.3939
P-value: 0.0000


/var/folders/p6/pvd52f996qs30t8lkkf2qh0r0000gn/T/ipykernel_36612/2671925888.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '*' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  markers[(p_matrix <= 0.05) & (p_matrix > 0.01)] = '*'    # p < 0.05
/var/folders/p6/pvd52f996qs30t8lkkf2qh0r0000gn/T/ipykernel_36612/2671925888.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '***' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  markers[p_matrix <= 0.001] = '***'                       # p < 0.001


In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('results/fixed-merged-subarticles-edges/importance-merged/unbalanced/total_df.csv', low_memory=False)

# All centrality measures
centralities = [
    'betweenness_centrality',
    'closeness_centrality', 
    'core_number',
    'degree_centrality',
    'disruption',
    'eigenvector_centrality',
    'harmonic_centrality',
    'hits_authority',
    'hits_hub',
    'in_degree_centrality',
    'out_degree_centrality',
    'pagerank',
    'relative_in_degree_centrality'
]

ground_truths = ['importance', 'doctypebranch']

# Calculate class distribution
print("Class Distribution:")
for gt in ground_truths:
    print(f"\n{gt.upper()} distribution:")
    print(df[gt].value_counts().sort_index())

# Calculate means and standard deviations
for gt in ground_truths:
    print(f"\n\nAnalysis for {gt}:")
    for measure in centralities:
        print(f"\n{measure}:")
        stats = df.groupby(gt)[measure].agg(['mean', 'std']).round(4)
        print(stats)

Class Distribution:

IMPORTANCE distribution:
importance
1     2150
2     5756
3    19895
Name: count, dtype: int64

DOCTYPEBRANCH distribution:
doctypebranch
1      514
2    20717
3     6570
Name: count, dtype: int64


Analysis for importance:

betweenness_centrality:
              mean     std
importance                
1           0.0003  0.0023
2           0.0001  0.0012
3           0.0000  0.0001

closeness_centrality:
              mean     std
importance                
1           0.0751  0.0569
2           0.0439  0.0433
3           0.0154  0.0310

core_number:
               mean     std
importance                 
1           15.6456  9.0058
2           14.6039  6.9237
3            6.0763  6.2456

degree_centrality:
              mean     std
importance                
1           0.0023  0.0048
2           0.0010  0.0013
3           0.0003  0.0005

disruption:
              mean     std
importance                
1           0.0935  0.3639
2           0.0301  0.1965
3      